# 150 — Observabilidad: logs, métricas y trazas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Observabilidad** = poder responder preguntas no anticipadas desde las salidas del
sistema. Tres señales:

- **Logs**: eventos discretos, estructurados (JSON), con `trace_id` para correlación.
- **Métricas**: series baratas — counter, gauge, histogram (→ percentiles). Regla: jamás
  ids de usuario como etiqueta (cardinalidad).
- **Trazas**: árbol de spans por petición; OpenTelemetry estandariza las tres señales y
  sus convenciones `gen_ai.*` (modelo, input/output tokens) para llamadas LLM.

Alertar sobre **síntomas** (SLO: p95, tasa de error, tasa de fallback), no sobre causas
(CPU). En IA, además: señales de calidad muestreadas y de costo por petición — porque el
fallo típico devuelve 200 con contenido degradado.


## 🧮 Ejemplo de referencia

Endpoint RAG, SLO p95 ≤ 2 000 ms; el p95 observado es 1 980 ms.

```text
span              p95        nota
retrieve_context    520 ms
llm_call          1 450 ms   ← 72 % de la contribución; input_tokens > 4 000 en la cola
guardrail_check      70 ms
```

Las trazas lentas comparten patrón (contextos > 4 000 tokens) → cap de contexto a 3 000
tokens → p95 total 1 630 ms. Los percentiles por span no se suman: solo la traza revela
cómo componen cada petición real.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("observability", seed=150)
show(result)


## Reflexión

1. ¿Por qué «responde 200 OK» es un criterio de salud casi vacío para un endpoint de LLM, y qué tres señales añadirías para que «sano» signifique algo?
2. Con histogramas por span cuyo p95 conoces, ¿por qué NO puedes calcular el p95 del total, y qué señal sí te lo da?
3. ¿Qué atributos de un span `gen_ai` usarías para detectar que un cambio de prompt duplicó el costo, y qué riesgo de privacidad introduce exportar el prompt completo?
